# Simulate Condition Models From Patient 1

This notebook generates per-condition simulated models by **scaling the segmented parts of patient 1**
using `models/condition_effects.json`, following the same scaling logic as `heart-scene.js`.

Notes:
- We do **not** scale `IVC`.
- Output is one OBJ per condition.


In [2]:
# Project root resolver
from pathlib import Path

PROJECT_MARKERS = [
    Path('models') / 'condition_effects.json',
    Path('heart_models') / 'patient_models',
    Path('data-processing'),
]

def find_project_root(start: Path) -> Path:
    cur = start.resolve()
    for parent in [cur] + list(cur.parents):
        if all((parent / marker).exists() for marker in PROJECT_MARKERS):
            return parent
    raise FileNotFoundError('Could not locate project root from ' + str(start))

PROJECT_ROOT = find_project_root(Path.cwd())
PROJECT_ROOT


PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026')

In [10]:
# Config
PATIENT_OBJ = PROJECT_ROOT / 'heart_models' / 'patient_models' / 'heart_model_pat1.obj'
CONDITION_JSON = PROJECT_ROOT / 'models' / 'condition_effects.json'
OUTPUT_DIR = PROJECT_ROOT / 'heart_models' / 'simulated_conditions'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATIENT_OBJ, CONDITION_JSON, OUTPUT_DIR


(PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/patient_models/heart_model_pat1.obj'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/models/condition_effects.json'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions'))

In [11]:
# Imports
import json
import math
import numpy as np


In [12]:
# Load condition effects
data = json.loads(CONDITION_JSON.read_text())
condition_multipliers = data.get('condition_multipliers', {})
conditions = sorted(condition_multipliers.keys())
print('Conditions:', len(conditions))
conditions[:10]


Conditions: 33


['AOPAAnastamosis',
 'ASD',
 'ArterialSwitch',
 'AtrialSwitch',
 'BilateralSVC',
 'CMRArtifactAO',
 'CMRArtifactPA',
 'CommonAtrium',
 'DIDORV',
 'DILV']

In [13]:
# Mapping + scaling logic (matches heart-scene.js)
PART_NAMES = ['LV', 'RV', 'LA', 'RA', 'AO', 'PA', 'PV', 'SVC']
PART_NAME_TO_VOL_KEY = {
    'LV': 'Label_1_vol_ml',
    'RV': 'Label_2_vol_ml',
    'LA': 'Label_3_vol_ml',
    'RA': 'Label_4_vol_ml',
    'AO': 'Label_5_vol_ml',
    'PA': 'Label_6_vol_ml',
    'PV': 'Label_7_vol_ml',
    'SVC': 'Label_8_vol_ml',
}

# OBJ group name -> part key
OBJ_NAME_TO_PART = {
    'Aorta': 'AO',
    'IVC': 'IVC',
    'SVC': 'SVC',
    'LV': 'LV',
    'RV': 'RV',
    'LA': 'LA',
    'RA': 'RA',
    'PA': 'PA',
    'PV': 'PV',
}

def compute_scale_for_condition(cond: str) -> dict:
    scales = {p: 1.0 for p in PART_NAMES}
    mults = condition_multipliers.get(cond, {})
    for part in PART_NAMES:
        vol_key = PART_NAME_TO_VOL_KEY[part]
        mult = mults.get(vol_key)
        if mult is None or mult <= 0:
            scales[part] = 1.0
            continue
        mult = max(0.2, min(5.0, float(mult)))
        scales[part] = mult ** (1.0 / 3.0)
    return scales


In [14]:
# OBJ parsing: per-group meshes with local vertices
def parse_obj_by_group(obj_path: Path):
    vertices = []  # list of [x,y,z]
    faces_by_group = {}  # group -> list of faces (vertex indices, 1-based)
    current_group = 'default'
    with obj_path.open() as f:
        for line in f:
            if line.startswith('v '):
                parts = line.strip().split()
                if len(parts) >= 4:
                    vertices.append([float(parts[1]), float(parts[2]), float(parts[3])])
            elif line.startswith('g '):
                current_group = line.strip().split(maxsplit=1)[1] if len(line.strip().split()) > 1 else 'default'
            elif line.startswith('f '):
                parts = line.strip().split()[1:]
                idxs = []
                for p in parts:
                    # supports v, v//n, v/t/n
                    v = p.split('/')[0]
                    idxs.append(int(v))
                faces_by_group.setdefault(current_group, []).append(idxs)
    return np.array(vertices, dtype=np.float32), faces_by_group

def build_group_meshes(vertices, faces_by_group):
    # Returns group -> (group_vertices, group_faces) with local indexing
    group_meshes = {}
    for group, faces in faces_by_group.items():
        used = sorted({i for face in faces for i in face})
        index_map = {old: new for new, old in enumerate(used, start=1)}
        group_verts = np.array([vertices[i-1] for i in used], dtype=np.float32)
        group_faces = [[index_map[i] for i in face] for face in faces]
        group_meshes[group] = (group_verts, group_faces)
    return group_meshes

def write_obj_groups(path: Path, group_meshes):
    # Writes each group with its own local vertices
    lines = []
    vert_offset = 0
    for group, (verts, faces) in group_meshes.items():
        lines.append(f'g {group}')
        for v in verts:
            lines.append(f'v {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}')
        for face in faces:
            # face indices are local, shift by vert_offset
            idxs = [str(i + vert_offset) for i in face]
            lines.append('f ' + ' '.join(idxs))
        vert_offset += len(verts)
    path.write_text('\n'.join(lines))


In [15]:
# Load patient 1 OBJ and split by groups
verts, faces_by_group = parse_obj_by_group(PATIENT_OBJ)
group_meshes = build_group_meshes(verts, faces_by_group)
print('Groups:', list(group_meshes.keys()))


Groups: ['LV', 'RV', 'LA', 'RA', 'Aorta', 'PA', 'SVC', 'IVC']


In [16]:
# Generate simulated models for all conditions
def scale_group_meshes(group_meshes, scales_by_part):
    out = {}
    for group, (gverts, gfaces) in group_meshes.items():
        part = OBJ_NAME_TO_PART.get(group, group)
        if part == 'IVC':
            s = 1.0
        else:
            s = scales_by_part.get(part, 1.0)
        out[group] = (gverts * float(s), gfaces)
    return out

written = []
for cond in conditions:
    scales = compute_scale_for_condition(cond)
    scaled_meshes = scale_group_meshes(group_meshes, scales)
    out_path = OUTPUT_DIR / f'simulated_{cond}.obj'
    write_obj_groups(out_path, scaled_meshes)
    written.append(out_path)

print('Wrote', len(written), 'files to', OUTPUT_DIR)
written[:5]


Wrote 33 files to /Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions


[PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions/simulated_AOPAAnastamosis.obj'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions/simulated_ASD.obj'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions/simulated_ArterialSwitch.obj'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions/simulated_AtrialSwitch.obj'),
 PosixPath('/Users/vehnilrangaraman/Desktop/COLLEGE/HACKLYTICS/2026/hacklytics2026/heart_models/simulated_conditions/simulated_BilateralSVC.obj')]